# Clase 5 — Unidad 5: Técnicas Básicas de Exploración de Datos Biomédicos

**Curso:** Introducción al Análisis de Datos — 2026, Segundo Semestre
**Material de referencia:** *Python Essentials for Biomedical Data Analysis* — Capítulo 5, "Basic Biomedical Data Exploration Techniques"

## Objetivos de aprendizaje
- Comprender qué es la **exploración de datos (EDA)** y por qué es el primer paso de todo análisis.
- Distinguir tipos de datos: **cuantitativos** (discretos/continuos) y **cualitativos** (nominales/ordinales).
- Conocer las propiedades básicas de un dataset: **forma, tamaño y dimensionalidad**.
- **Inspeccionar y limpiar** datos: `head`/`tail`/`sample`, faltantes, errores y conversión de tipos.
- Analizar **datos categóricos** con tablas de frecuencia y de contingencia, y visualizarlos.
- Realizar una **exploración inicial** con estadísticas descriptivas y gráficos (histograma, boxplot, dispersión).

## Contenidos de la clase
1. ¿Qué es la exploración de datos?
2. Entender el dataset (tipos de datos, forma y tamaño)
3. Inspección y limpieza de datos
4. Datos categóricos
5. Exploración inicial en la práctica
6. Preparación para el análisis avanzado
7. Ejercicios de práctica

> Esta clase corresponde a la **Unidad 5** del libro guía. Usaremos un conjunto de datos sintético de pacientes que incluye, a propósito, **valores imposibles** y una columna guardada como **texto**, para practicar la detección de errores y la conversión de tipos.

## Preparación del entorno

Instalamos **pandas** y **numpy** (manejo de datos) y **matplotlib** (visualización). Luego creamos el dataset `base`.

In [ ]:
%pip install pandas numpy matplotlib

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(7)
n = 150

base = pd.DataFrame({
    "paciente_id": [f"P{i:03d}" for i in range(1, n + 1)],
    "edad": np.random.randint(18, 90, n),
    "sexo": np.random.choice(["F", "M"], n),
    "tipo_sangre": np.random.choice(["A", "B", "AB", "O"], n, p=[0.40, 0.10, 0.05, 0.45]),
    "etapa_cancer": np.random.choice(["I", "II", "III", "IV"], n, p=[0.40, 0.30, 0.20, 0.10]),
    "glucosa": np.random.normal(100, 20, n).round(1),
    "colesterol": np.random.normal(190, 30, n).round(1),
    "imc": np.random.normal(26, 4, n).round(1),
    "diagnostico": np.random.choice(["Sano", "Prediabetes", "Diabetes"], n, p=[0.60, 0.25, 0.15]),
})

# Peso guardado como TEXTO (para practicar la conversión de tipos más adelante)
base["peso_kg_texto"] = np.random.normal(70, 12, n).round(1).astype(str)

# Inyectar valores IMPOSIBLES (errores de registro) para detectarlos luego
base.loc[5, "edad"] = -3      # edad negativa: imposible
base.loc[8, "glucosa"] = 0    # glucosa 0: imposible

# Inyectar valores faltantes (en filas 20+ para no borrar los errores inyectados)
rng = np.random.default_rng(1)
for col in ["glucosa", "colesterol", "imc"]:
    base.loc[rng.choice(range(20, n), size=10, replace=False), col] = np.nan

print("Dataset creado:", base.shape)
base.head()

## 1. ¿Qué es la exploración de datos?

La **exploración de datos** (o *EDA, Exploratory Data Analysis*) es el **primer paso** de todo análisis. Consiste en examinar el dataset para entender su estructura, características y patrones **antes** de aplicar técnicas avanzadas.

Es como el trabajo preliminar de un detective: buscar pistas antes de resolver el caso. Nos permite:
- Entender la **estructura y calidad** de los datos.
- Detectar **problemas**: faltantes, outliers, errores, inconsistencias.
- Identificar **patrones y relaciones** iniciales.
- **Formular hipótesis** y elegir los métodos de análisis adecuados.

**El proceso de exploración**, paso a paso:
1. Inspección inicial (tamaño, estructura, tipos de datos)
2. Limpieza de datos (faltantes, errores, outliers)
3. Estadísticas descriptivas
4. Visualización
5. Identificación de relaciones y anomalías
6. Formulación preliminar de hipótesis
7. Preparación para el análisis avanzado

## 2. Entender el dataset

### 2.1 Tipos de datos: cuantitativos vs cualitativos

| Categoría | Subtipo | Descripción | Ejemplo biomédico |
|---|---|---|---|
| **Cuantitativos** (numéricos) | **Discretos** | Valores contables (enteros). | Número de pacientes, divisiones celulares. |
| | **Continuos** | Cualquier valor dentro de un rango. | Temperatura corporal, actividad enzimática. |
| **Cualitativos** (categóricos) | **Nominales** | Categorías sin orden. | Tipo de sangre (A, B, AB, O), sexo. |
| | **Ordinales** | Categorías con orden. | Etapa de cáncer (I, II, III, IV), escala de dolor. |

En Pandas, el **tipo de dato** (`dtype`) de cada columna nos orienta: `int64`/`float64` son numéricos, `object` suele ser texto/categórico.

In [ ]:
datos = base.copy()

# Ver el tipo de dato de cada columna
print("Tipos de dato por columna:")
print(datos.dtypes)

# Separar columnas numéricas y categóricas
numericas = datos.select_dtypes(include="number").columns.tolist()
categoricas = datos.select_dtypes(include="object").columns.tolist()
print("\nNuméricas:", numericas)
print("Categóricas (object):", categoricas)

### 2.2 Forma, tamaño y dimensionalidad

- **Forma (shape):** número de filas y columnas → `(filas, columnas)`.
- **Tamaño (size):** total de datos = filas × columnas.
- **Dimensionalidad:** número de variables (columnas).

In [ ]:
datos = base.copy()

print("Forma (filas, columnas):", datos.shape)
print("Tamaño (total de celdas):", datos.size)
print("Dimensionalidad (n° de variables):", datos.shape[1])
print("Número de pacientes (filas):", len(datos))

## 3. Inspección y limpieza de datos

### 3.1 Inspección básica: `head`, `tail` y muestra aleatoria

Un buen punto de partida es mirar las **primeras** filas (`head`), las **últimas** (`tail`) y una **muestra aleatoria** (`sample`), útil en datasets grandes para detectar irregularidades que no se ven al inicio o al final.

In [ ]:
datos = base.copy()

# Primeras filas
print("Primeras filas (head):")
print(datos.head())

In [ ]:
# Últimas filas
print("Últimas filas (tail):")
print(datos.tail(3))

In [ ]:
# Muestra aleatoria de 5 filas
print("Muestra aleatoria (sample):")
print(datos.sample(5, random_state=42))

In [ ]:
# Resumen de estructura: tipos y no-nulos por columna
datos.info()

# Estadísticas descriptivas de las variables numéricas
print("\nEstadísticas descriptivas (numéricas):")
print(datos.describe().round(1))

### 3.2 Identificar valores faltantes

Con Pandas, `isnull()` e `info()` permiten detectar rápidamente los faltantes (repasamos lo visto en la Unidad 4).

In [ ]:
datos = base.copy()

print("Faltantes por columna:")
print(datos.isnull().sum())

print("\nPorcentaje de faltantes (%):")
print((datos.isnull().mean() * 100).round(1))

### 3.3 Detectar y corregir errores o anomalías

Además de los faltantes, hay que buscar **valores imposibles** (fuera de rangos biológicos plausibles), como edades negativas o glucosa igual a cero. Aquí es clave el **conocimiento del dominio**.

In [ ]:
datos = base.copy()

# Buscar valores biológicamente imposibles
edades_invalidas = datos[datos["edad"] <= 0]
glucosa_invalida = datos[datos["glucosa"] <= 0]

print("Pacientes con edad imposible (<= 0):")
print(edades_invalidas[["paciente_id", "edad"]])
print("\nPacientes con glucosa imposible (<= 0):")
print(glucosa_invalida[["paciente_id", "glucosa"]])

In [ ]:
# Corregir: reemplazar los valores imposibles por NaN (para tratarlos como faltantes)
datos.loc[datos["edad"] <= 0, "edad"] = np.nan
datos.loc[datos["glucosa"] <= 0, "glucosa"] = np.nan

print("Edad mínima tras la corrección:", datos["edad"].min())
print("Glucosa mínima tras la corrección:", datos["glucosa"].min())

### 3.4 Conversión de tipos de datos

A veces un número queda guardado como **texto** (`object`), lo que impide hacer cálculos. `astype()` convierte al tipo correcto. Para variables categóricas, `get_dummies()` hace one-hot encoding (visto en la Unidad 4).

In [ ]:
datos = base.copy()

# La columna 'peso_kg_texto' está guardada como texto (object)
print("Tipo antes:", datos["peso_kg_texto"].dtype)
print("¿Se puede promediar el texto? ->", end=" ")
try:
    datos["peso_kg_texto"].mean()
except TypeError as e:
    print("No:", e)

# Convertir a número
datos["peso_kg"] = datos["peso_kg_texto"].astype(float)
print("Tipo después:", datos["peso_kg"].dtype)
print("Peso promedio:", round(datos["peso_kg"].mean(), 1), "kg")

## 4. Datos categóricos

Las variables categóricas (nominales y ordinales) son muy frecuentes en biomedicina (tipo de sangre, diagnóstico, etapa de cáncer). Se analizan con **tablas de frecuencia** y **tablas de contingencia**, y se visualizan con **gráficos de barras y de torta**.

### 4.1 Tabla de frecuencias

`value_counts()` cuenta cuántas veces aparece cada categoría; con `normalize=True` obtenemos frecuencias relativas (proporciones).

In [ ]:
datos = base.copy()

# Tabla de frecuencias del diagnóstico
print("Frecuencia absoluta:")
print(datos["diagnostico"].value_counts())

print("\nFrecuencia relativa (%):")
print((datos["diagnostico"].value_counts(normalize=True) * 100).round(1))

### 4.2 Tabla de contingencia (cross-tabulation)

`pd.crosstab()` cruza **dos** variables categóricas para examinar su relación. Es la base de pruebas estadísticas como chi-cuadrado (que verás más adelante).

In [ ]:
# Relación entre sexo y diagnóstico
tabla = pd.crosstab(datos["sexo"], datos["diagnostico"])
print("Tabla de contingencia (sexo x diagnóstico):")
print(tabla)

# Con totales por fila y columna
print("\nCon totales (márgenes):")
print(pd.crosstab(datos["sexo"], datos["diagnostico"], margins=True, margins_name="Total"))

### 4.3 Visualización de datos categóricos

In [ ]:
import matplotlib.pyplot as plt

# Gráfico de barras: frecuencia de cada diagnóstico
conteo = datos["diagnostico"].value_counts()

plt.figure(figsize=(6, 4))
plt.bar(conteo.index, conteo.values, color=["#59A14F", "#F1A340", "#E15759"])
plt.title("Distribución de diagnósticos")
plt.xlabel("Diagnóstico")
plt.ylabel("N° de pacientes")
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de torta: proporción de tipos de sangre
conteo_sangre = datos["tipo_sangre"].value_counts()

plt.figure(figsize=(5, 5))
plt.pie(conteo_sangre.values, labels=conteo_sangre.index, autopct="%1.1f%%", startangle=90)
plt.title("Proporción de tipos de sangre")
plt.tight_layout()
plt.show()

## 5. Exploración inicial en la práctica

Un **enfoque sistemático** combina estadísticas descriptivas (media, mediana, desviación) con visualizaciones que revelan patrones no evidentes en los números crudos:
- **Histograma:** distribución de una variable numérica.
- **Boxplot:** dispersión y outliers.
- **Diagrama de dispersión (scatter):** relación entre dos variables.

Primero preparamos un conjunto limpio (corrigiendo los valores imposibles).

In [ ]:
# Conjunto de trabajo: corregimos los valores imposibles a NaN
eda = base.copy()
eda.loc[eda["edad"] <= 0, "edad"] = np.nan
eda.loc[eda["glucosa"] <= 0, "glucosa"] = np.nan

# Estadísticas de resumen de algunas variables clave
print(eda[["edad", "glucosa", "colesterol", "imc"]].describe().round(1))

In [ ]:
import matplotlib.pyplot as plt

# Histograma: distribución de la edad
plt.figure(figsize=(6, 4))
plt.hist(eda["edad"].dropna(), bins=15, color="#4C78A8", edgecolor="white")
plt.title("Distribución de la edad")
plt.xlabel("Edad (años)")
plt.ylabel("N° de pacientes")
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot: dispersión y outliers de la glucosa y el colesterol
plt.figure(figsize=(6, 4))
plt.boxplot([eda["glucosa"].dropna(), eda["colesterol"].dropna()])
plt.xticks([1, 2], ["Glucosa", "Colesterol"])   # etiquetas de cada caja
plt.title("Boxplot de glucosa y colesterol")
plt.ylabel("mg/dL")
plt.tight_layout()
plt.show()

In [ ]:
# Diagrama de dispersión: relación entre IMC y glucosa
sub = eda.dropna(subset=["imc", "glucosa"])

plt.figure(figsize=(6, 4))
plt.scatter(sub["imc"], sub["glucosa"], alpha=0.6, color="#B07AA1")
plt.title("Relación entre IMC y glucosa")
plt.xlabel("IMC (kg/m²)")
plt.ylabel("Glucosa (mg/dL)")
plt.tight_layout()
plt.show()

### Errores comunes a evitar (*pitfalls*)

- **Ignorar la calidad de los datos:** no revisar faltantes, outliers o errores antes de analizar.
- **Sesgo del analista:** dejar que las expectativas influyan en la interpretación.
- **Ignorar el contexto:** los datos biomédicos no existen en el vacío; hay que considerar el contexto clínico, demográfico y experimental.

## 6. Preparación para el análisis avanzado

La exploración inicial prepara el terreno para técnicas más complejas. Conviene **resumir los hallazgos**:

- **Distribución de los datos:** ¿son normales o sesgados?
- **Variables clave:** ¿cuáles parecen más informativas?
- **Posibles relaciones:** ¿se observaron correlaciones o asociaciones?
- **Anomalías y patrones:** ¿hubo outliers, tendencias inesperadas o faltantes relevantes?

Con esto se pueden **formular hipótesis** claras y comprobables (ej. "¿la glucosa se asocia con el diagnóstico de diabetes?") que guiarán el análisis avanzado: pruebas de hipótesis, modelos predictivos (machine learning), análisis de supervivencia, etc. (temas de las próximas unidades).

## 7. Ejercicios de práctica

Usa el conjunto `base` para resolver:

1. **Inspección:** muestra la forma, el tamaño y los tipos de dato de `base`. ¿Cuántas columnas son numéricas y cuántas categóricas?
2. **head/tail/sample:** muestra las primeras 8 filas, las últimas 4 y una muestra aleatoria de 6.
3. **Faltantes:** ¿qué columna tiene más valores faltantes? Muestra el porcentaje por columna.
4. **Errores:** encuentra e imprime los pacientes con valores imposibles en `edad` o `glucosa`, y corrígelos a `NaN`.
5. **Conversión de tipos:** convierte `peso_kg_texto` a numérico y calcula su media y desviación estándar.
6. **Frecuencias:** crea la tabla de frecuencias (absoluta y relativa) de `etapa_cancer`.
7. **Contingencia:** cruza `etapa_cancer` con `diagnostico` en una tabla de contingencia con totales.
8. **Visualización:** haz un histograma del `colesterol` y un gráfico de barras de `tipo_sangre`.

### Preguntas para reflexionar
1. ¿Por qué la exploración de datos es un paso imprescindible antes del análisis avanzado?
2. ¿Cuál es la diferencia entre datos nominales y ordinales? Da un ejemplo clínico de cada uno.
3. ¿Qué información entrega una tabla de contingencia que no da una tabla de frecuencias simple?
4. ¿Por qué un valor numérico guardado como texto puede arruinar un análisis?
5. ¿Qué gráfico usarías para detectar outliers y por qué?

In [ ]:
# Espacio para resolver los ejercicios
